<a href="https://colab.research.google.com/github/peacewhile/PHM-Learning-Task/blob/main/NGAFID_DATASET_MINIROCKET_EXAMPLE6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)

!pip install tsai -q

In [ ]:
!pip install --upgrade ipython -q

In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

In [ ]:
%load_ext autoreload

In [ ]:
%autoreload
from tsai.basics import *
from tsai.models.MINIROCKET_Pytorch import *
from tsai.models.utils import *
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xzf /content/drive/MyDrive/NGAFIDDATASET/2days.tar.gz -C /content/

In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

import numpy as np
import pandas as pd
import pickle

# 手动构建一个简单的 dm 对象
class SimpleDM:
    pass

dm = SimpleDM()

# 注入元数据
dm.flight_header_df = pd.read_csv(
    '/content/2days/flight_header.csv',
    index_col='Master Index'
)

# 注入时序数据
with open('/content/2days/flight_data.pkl', 'rb') as f:
    dm.data_dict = pickle.load(f)

# 加载归一化参数
stats = pd.read_csv('/content/2days/stats.csv', index_col=0)
print(stats.head())
print(stats.shape)

In [ ]:
import numpy as np
stats_numeric = stats.drop(columns=['cluster'])
dm.mins = stats_numeric.loc[1].values.astype(np.float32)
dm.maxs = stats_numeric.loc[0].values.astype(np.float32)

In [ ]:
import types

def get_numpy_dataset(self, fold, training):
    if training:
        mask = self.flight_header_df['fold'] != fold
    else:
        mask = self.flight_header_df['fold'] == fold
    subset = self.flight_header_df[mask]
    data = [self.data_dict[idx] for idx in subset.index]
    return {
        'data':         data,
        'target_class': subset['target_class'].values,
        'before_after': subset['before_after'].values,
        'fold':         subset['fold'].values
    }

dm.get_numpy_dataset = types.MethodType(get_numpy_dataset, dm)

In [ ]:
from fastai.callback.progress import CSVLogger

from tqdm.autonotebook import tqdm

In [ ]:
def pad_or_truncate(data_list, target_len=3000):
    result = []
    for arr in data_list:
        T = arr.shape[0]
        if T >= target_len:
            arr = arr[:target_len, :]
        else:
            pad = np.zeros((target_len - T, arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        result.append(arr)
    return np.array(result, dtype=np.float32).transpose(0, 2, 1)

In [ ]:
import gc

save_path = '.'
model_name = 'MINIROCKET_BINARY'

mins23 = dm.mins[:23].reshape(-1, 1)
maxs23 = dm.maxs[:23].reshape(-1, 1)

for fold in tqdm(range(5)):

    save_filename = save_path + '%s_%i' % (model_name, fold)

    train_dict = dm.get_numpy_dataset(fold=fold, training=True)
    test_dict  = dm.get_numpy_dataset(fold=fold, training=False)

    train_X = pad_or_truncate(train_dict['data'], target_len=200)  # ← 改为200
    train_X = (train_X - mins23) / (maxs23 - mins23)
    train_X = np.nan_to_num(train_X, copy=False)

    test_X = pad_or_truncate(test_dict['data'], target_len=200)    # ← 改为200
    test_X = (test_X - mins23) / (maxs23 - mins23)
    test_X = np.nan_to_num(test_X, copy=False)

    train_Y = np.array(train_dict['before_after'])
    test_Y  = np.array(test_dict['before_after'])

    splits = [list(np.arange(len(train_Y)))]
    splits.append(list(np.arange(len(test_Y)) + len(train_Y)))

    torch.cuda.empty_cache()
    mrf = MiniRocketFeatures(train_X.shape[1], train_X.shape[2]).to(default_device())
    chunksize = 32                                                  # ← 从64改为32

    mrf.fit(train_X, chunksize=chunksize)

    X_feat = get_minirocket_features(np.concatenate([train_X, test_X]), mrf, chunksize=chunksize, to_np=True)

    Y = np.concatenate([train_Y, test_Y])

    PATH = Path("./models/MRF.pt")
    PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mrf.state_dict(), PATH)

    tfms = [None, TSClassification()]
    batch_tfms = TSStandardize(by_sample=True)
    dls = get_ts_dls(X_feat, Y, splits=splits, tfms=tfms, batch_tfms=batch_tfms)
    model = build_ts_model(MiniRocketHead, dls=dls)

    learn = Learner(dls, model, metrics=accuracy, cbs=[ShowGraph(), CSVLogger(save_filename)])

    results = learn.fit_one_cycle(200, 2.5e-5)

    # 每折结束后释放内存
    del train_X, test_X, X_feat, mrf, model, learn, dls, results
    torch.cuda.empty_cache()
    gc.collect()